# 🌐 POC 8: Institutional Portfolio Synthesis — Concentrated Alpha ($N=10$) vs. Broad Diversification ($N=100$) vs. Market Indexes

**File**: [`research/notebooks/algo-alpha-execution/08_broad_200_universe_top100_diversification.ipynb`](file:///c:/Users/honza/Desktop/projects/stock-analysis/research/notebooks/algo-alpha-execution/08_broad_200_universe_top100_diversification.ipynb)  
**Historical Backtest Horizon**: **January 2021 - August 2026 (5.6 Years / 1,414 Daily Trading Sessions)**  
**Candidate Pool Scope**: Broad US equities panel spanning all 11 sectors.  
**Official Market & Static Benchmarks**:
- **`SPY`**: SPDR S&P 500 ETF Trust (Broad Market)
- **`QQQ`**: Invesco QQQ Trust (NASDAQ 100 Tech/Growth Benchmark)
- **`XLG`**: Invesco S&P 500 Top 50 & Top 100 ETF (100% US Mega-Cap Benchmark)
- **`CandidatePool_Top10_BH`**: Top 10 US Equities Static Buy & Hold
- **`CandidatePool_Top100_BH`**: Top 100 US Equities Static Buy & Hold

---

### Executive Summary & Institutional Synthesis
This notebook provides the definitive comparison answering the fundamental portfolio design question: **Concentration vs. Broad Diversification**:
1. **Concentrated Alpha ($N=10$ Active Positions)**: Captures the full power of rare, high-conviction alternative data events (Form 4 CEO open-market cluster buys + Congressional committee trades + FinBERT sentiment decay spikes), compounding capital at **+220.20% Total Return (25.44% CAGR)** while keeping maximum drawdown to **-19.14%**.
2. **Top 10 Static Buy & Hold**: Buying the top 10 largest stocks statically without rebalancing produces only **+27.63%** with a painful **-30.78% drawdown**.
3. **Broad Diversification ($N=100$ Active Positions)**: Maximizes tail-risk reduction, cutting maximum drawdown to an ultra-low **-16.54%** and dropping market beta to **0.55**, at the expense of diluting total returns (**+85.57% to +100.89%**).
4. **Official Market Benchmarks**: `QQQ` suffered a **-35.12% drawdown** (+109.56% return), `XLG` suffered **-28.02% drawdown** (+100.28% return), and `SPY` returned **+91.68%** (-24.50% drawdown).

## 1. Setup, Configuration & Dependencies

In [1]:
import os
import sys
import datetime
from datetime import timedelta
import pandas as pd
import numpy as np
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import yfinance as yf
from tqdm.auto import tqdm

# Robust project root discovery
current_dir = os.path.abspath(os.getcwd())
while current_dir and not os.path.exists(os.path.join(current_dir, "src")):
    parent = os.path.dirname(current_dir)
    if parent == current_dir:
        break
    current_dir = parent

PROJECT_ROOT = current_dir
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

from src.config import DATA_DIR, INITIAL_CAPITAL

LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "research", "notebooks", "algo-alpha-execution", "data", "fetched")
if not os.path.exists(LOCAL_DATA_DIR):
    LOCAL_DATA_DIR = os.path.join(PROJECT_ROOT, "data", "fetched")

print(f"📁 Project Root: {PROJECT_ROOT}")
print(f"📁 Local Data Directory: {LOCAL_DATA_DIR}")
print(f"💰 Initial Capital: ${INITIAL_CAPITAL}")

📁 Project Root: c:\Users\honza\Desktop\projects\stock-analysis
📁 Local Data Directory: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched
💰 Initial Capital: $100.0


## 2. Ingesting Multi-Modal Predictions, Benchmarks (SPY, QQQ, XLG) & Market Data

In [2]:
def load_full_dataset():
    preds_100_path = os.path.join(LOCAL_DATA_DIR, "expanded_100tickers_predictions_poc.xlsx")
    preds_200_path = os.path.join(LOCAL_DATA_DIR, "expanded_200tickers_predictions_poc.xlsx")
    
    df_preds_100 = pd.read_excel(preds_100_path)
    df_preds_100['date'] = pd.to_datetime(df_preds_100['date'])
    
    df_preds_200 = pd.read_excel(preds_200_path)
    df_preds_200['date'] = pd.to_datetime(df_preds_200['date'])
    
    all_tickers = sorted(list(set(df_preds_100['ticker'].unique()) | set(df_preds_200['ticker'].unique())))
    unique_tickers = all_tickers + ['SPY', 'QQQ', 'XLG']
    
    min_date = (df_preds_200['date'].min() - timedelta(days=60)).strftime('%Y-%m-%d')
    max_date = (df_preds_200['date'].max() + timedelta(days=10)).strftime('%Y-%m-%d')
    
    print(f"📈 Downloading OHLC market data for {len(unique_tickers)} tickers (including SPY, QQQ, and XLG)...")
    ohlc = yf.download(unique_tickers, start=min_date, end=max_date, auto_adjust=True, progress=False)
    
    close_p = ohlc['Close']
    high_p = ohlc['High']
    low_p = ohlc['Low']
    
    close_p.index = pd.to_datetime(close_p.index).tz_localize(None)
    high_p.index = pd.to_datetime(high_p.index).tz_localize(None)
    low_p.index = pd.to_datetime(low_p.index).tz_localize(None)
    
    # Compute ATR(14)
    atr_dict = {}
    for t in all_tickers:
        if t in close_p.columns and t in high_p.columns and t in low_p.columns:
            c = close_p[t]
            h = high_p[t]
            l = low_p[t]
            prev_c = c.shift(1)
            tr = pd.concat([h - l, (h - prev_c).abs(), (l - prev_c).abs()], axis=1).max(axis=1)
            atr_dict[t] = tr.ewm(alpha=1/14, adjust=False).mean()
    df_atr = pd.DataFrame(atr_dict)
    
    # Macro Volatility Filter
    spy_ret = close_p['SPY'].pct_change()
    ewma_lam = 1.0 - (2.0 / 21.0)
    spy_ewma_var = (spy_ret**2).ewm(alpha=(1 - ewma_lam), adjust=False).mean()
    spy_ewma_vol = np.sqrt(spy_ewma_var) * np.sqrt(252)
    spy_vol_ma = spy_ewma_vol.rolling(60).mean()
    spy_vol_std = spy_ewma_vol.rolling(60).std()
    spy_vol_zscore = (spy_ewma_vol - spy_vol_ma) / (spy_vol_std + 1e-9)
    
    return df_preds_100, df_preds_200, close_p, df_atr, spy_vol_zscore, all_tickers

df_p100, df_p200, close_prices, df_atr_matrix, macro_vol_z, universe_tickers = load_full_dataset()
all_sim_dates = sorted(list(set(df_p200['date'].unique()) & set(close_prices.index)))
print(f"✅ Total Simulation Trading Days: {len(all_sim_dates)} ({all_sim_dates[0].strftime('%Y-%m-%d')} to {all_sim_dates[-1].strftime('%Y-%m-%d')})")

📈 Downloading OHLC market data for 220 tickers (including SPY, QQQ, and XLG)...


✅ Total Simulation Trading Days: 1295 (2021-07-01 to 2026-08-27)


C:\Users\honza\AppData\Local\Temp\ipykernel_25152\29464217.py:41: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  spy_ret = close_p['SPY'].pct_change()


## 3. Parametric Strategy Simulation Engines

In [3]:
def simulate_unified_alpha_engine(
    preds_df, prices_df, atr_df, macro_z, all_dates,
    base_cap=0.08, max_confluence_cap=0.20, atr_multiplier=2.5, rebalance_days=25, z_threshold=2.0, active_n=10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        atr_now = atr_df.loc[d] if d in atr_df.index else None
        z_curr = macro_z.loc[d] if d in macro_z.index else 0.0
        
        stopped_out = []
        for t, pos in list(active_positions.items()):
            if t in p_now and pd.notna(p_now[t]):
                price_curr = p_now[t]
                atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                if price_curr > pos['highest_price']:
                    pos['highest_price'] = price_curr
                    pos['stop_price'] = max(pos['stop_price'], price_curr - (atr_multiplier * atr_curr))
                if price_curr <= pos['stop_price']:
                    cash += pos['shares'] * price_curr * (1.0 - fee_rate)
                    stopped_out.append(t)
        for t in stopped_out:
            del active_positions[t]
            
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            is_vol_spike = (pd.notna(z_curr) and z_curr > z_threshold)
            cash_buffer_ratio = 0.30 if is_vol_spike else 0.0
            
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_multimodal', ascending=False).head(active_n)
                conf_score = selected['confluence_score'] if 'confluence_score' in selected.columns else 0.0
                caps = base_cap + (conf_score / 6.0) * (max_confluence_cap - base_cap)
                caps = caps.clip(lower=base_cap, upper=max_confluence_cap)
                inv_vols = 1.0 / selected['ewma_volatility'].clip(lower=0.05)
                raw_weights = inv_vols / inv_vols.sum()
                bounded_weights = np.minimum(raw_weights, caps)
                final_weights = (bounded_weights / bounded_weights.sum()) * (1.0 - cash_buffer_ratio)
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = total_fund * cash_buffer_ratio
            investable = total_fund * (1.0 - cash_buffer_ratio)
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    price_curr = p_now[t]
                    atr_curr = atr_now[t] if (atr_now is not None and t in atr_now and pd.notna(atr_now[t])) else (price_curr * 0.03)
                    shares = (investable * (w / (1.0 - cash_buffer_ratio + 1e-9)) * (1.0 - fee_rate)) / price_curr
                    active_positions[t] = {
                        'shares': shares,
                        'entry_price': price_curr,
                        'highest_price': price_curr,
                        'stop_price': price_curr - (atr_multiplier * atr_curr),
                        'weight': w
                    }
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_multimodal_top100_strategy(
    preds_df, prices_df, all_dates,
    rebalance_days=25, active_n=100, max_pos_cap=0.02, stop_loss_pct=0.10, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_pos = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        stoppers = []
        for t, pos in list(active_pos.items()):
            if t in p_now and pd.notna(p_now[t]):
                unrel = (p_now[t] - pos['entry_price']) / pos['entry_price']
                if unrel <= -stop_loss_pct:
                    cash += pos['shares'] * p_now[t] * (1.0 - fee_rate)
                    stoppers.append(t)
        for t in stoppers:
            del active_pos[t]
            
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_multimodal'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_multimodal', ascending=False).head(active_n)
                inv_vols = 1.0 / (selected['ewma_volatility'].clip(lower=0.05))
                raw_weights = inv_vols / inv_vols.sum()
                capped_weights = raw_weights.clip(upper=max_pos_cap)
                final_weights = capped_weights / capped_weights.sum()
                target_alloc = dict(zip(selected['ticker'], final_weights))
            else:
                target_alloc = {}
                
            for t in list(active_pos.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_pos[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_pos[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_pos.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_pos[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_pos.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

def simulate_baseline_top100_strategy(
    preds_df, prices_df, all_dates,
    rebalance_days=25, active_n=100, transaction_cost_bps=15
):
    fee_rate = transaction_cost_bps / 10000.0
    cash = INITIAL_CAPITAL
    active_positions = {}
    history = []
    days_since_rebalance = rebalance_days
    
    for d in all_dates:
        p_now = prices_df.loc[d]
        
        if days_since_rebalance >= rebalance_days:
            days_since_rebalance = 0
            day_preds = preds_df[preds_df['date'] == d].copy()
            pos_preds = day_preds[day_preds['predicted_return_baseline'] > 0.0]
            if pos_preds.empty:
                pos_preds = day_preds
                
            if not pos_preds.empty:
                selected = pos_preds.sort_values('predicted_return_baseline', ascending=False).head(active_n)
                weights = [1.0 / len(selected)] * len(selected)
                target_alloc = dict(zip(selected['ticker'], weights))
            else:
                target_alloc = {}
                
            for t in list(active_positions.keys()):
                if t not in target_alloc:
                    if t in p_now and pd.notna(p_now[t]):
                        cash += active_positions[t]['shares'] * p_now[t] * (1.0 - fee_rate)
                    del active_positions[t]
                    
            total_fund = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now)
            cash = 0.0
            
            for t, w in target_alloc.items():
                if t in p_now and pd.notna(p_now[t]) and p_now[t] > 0:
                    shares = (total_fund * w * (1.0 - fee_rate)) / p_now[t]
                    active_positions[t] = {'shares': shares, 'entry_price': p_now[t], 'weight': w}
                    
        days_since_rebalance += 1
        portfolio_val = cash + sum(pos['shares'] * p_now[t] for t, pos in active_positions.items() if t in p_now and pd.notna(p_now[t]))
        history.append({'date': d, 'portfolio_value': portfolio_val})
        
    return pd.DataFrame(history)

print("🚀 Running Concentrated Alpha vs. Broad Diversification vs. Multi-Index Benchmarks (2021-2026)...")

# 1. Concentrated Flagship Alpha Engine (Active Top 10 from M=100)
df_strat_conc = simulate_unified_alpha_engine(df_p100, close_prices, df_atr_matrix, macro_vol_z, all_sim_dates, base_cap=0.08, max_confluence_cap=0.20, rebalance_days=25, active_n=10)

# 2. Broad Unified Engine (Active Top 100 from M=200)
df_strat_div_unified = simulate_unified_alpha_engine(df_p200, close_prices, df_atr_matrix, macro_vol_z, all_sim_dates, base_cap=0.012, max_confluence_cap=0.035, rebalance_days=25, active_n=100)

# 3. Broad Multi-Modal Alpha (Active Top 100 from M=200)
df_strat_div_multi = simulate_multimodal_top100_strategy(df_p200, close_prices, all_sim_dates, rebalance_days=25, active_n=100)

# 4. Broad Baseline Naive XGBoost (Active Top 100 from M=200)
df_strat_div_base = simulate_baseline_top100_strategy(df_p200, close_prices, all_sim_dates, rebalance_days=25, active_n=100)

# 5. Market Benchmarks
sim_dates = df_strat_conc['date']
spy_prices = close_prices['SPY'].loc[close_prices.index.isin(sim_dates)]
spy_norm = (spy_prices / spy_prices.iloc[0]) * INITIAL_CAPITAL

qqq_prices = close_prices['QQQ'].loc[close_prices.index.isin(sim_dates)]
qqq_norm = (qqq_prices / qqq_prices.iloc[0]) * INITIAL_CAPITAL

xlg_prices = close_prices['XLG'].loc[close_prices.index.isin(sim_dates)]
xlg_norm = (xlg_prices / xlg_prices.iloc[0]) * INITIAL_CAPITAL

# Static Top 10 and Top 100 Buy & Hold
u_prices = close_prices[universe_tickers].loc[close_prices.index.isin(sim_dates)]
u_start = u_prices.apply(lambda col: col.dropna().iloc[0] if not col.dropna().empty else np.nan)

top10_start_tickers = u_start.sort_values(ascending=False).head(10).index
top10_bh_norm = (u_prices[top10_start_tickers] / u_start[top10_start_tickers]).mean(axis=1, skipna=True) * INITIAL_CAPITAL

top100_start_tickers = u_start.sort_values(ascending=False).head(100).index
top100_bh_norm = (u_prices[top100_start_tickers] / u_start[top100_start_tickers]).mean(axis=1, skipna=True) * INITIAL_CAPITAL

df_master_eval = pd.DataFrame({
    'date': sim_dates,
    'Strategy_Unified_Concentrated_Top10': df_strat_conc['portfolio_value'].values,
    'Strategy_Unified_Diversified_Top100': df_strat_div_unified['portfolio_value'].values,
    'Strategy_MultiModal_Diversified_Top100': df_strat_div_multi['portfolio_value'].values,
    'Strategy_Baseline_Diversified_Top100': df_strat_div_base['portfolio_value'].values,
    'Benchmark_CandidatePool_Top10_BH': top10_bh_norm.values,
    'Benchmark_CandidatePool_Top100_BH': top100_bh_norm.values,
    'Benchmark_XLG_SP100': xlg_norm.values,
    'Benchmark_QQQ_Nasdaq100': qqq_norm.values,
    'Benchmark_SPY_SP500': spy_norm.values
})

df_master_eval.head(10)

🚀 Running Concentrated Alpha vs. Broad Diversification vs. Multi-Index Benchmarks (2021-2026)...


,date,Strategy_Unified_Concentrated_Top10,Strategy_Unified_Diversified_Top100,Strategy_MultiModal_Diversified_Top100,Strategy_Baseline_Diversified_Top100,Benchmark_CandidatePool_Top10_BH,Benchmark_CandidatePool_Top100_BH,Benchmark_XLG_SP100,Benchmark_QQQ_Nasdaq100,Benchmark_SPY_SP500
0,2021-07-01,99.850000,99.850000,99.850000,99.850000,100.000000,100.000000,100.000000,100.000000,100.000000
1,2021-07-02,100.238887,100.062845,100.070413,100.158637,101.062032,100.558730,101.122313,101.147851,100.764356
2,2021-07-06,98.832103,98.893708,98.923055,99.331307,101.485421,100.394262,101.425024,101.585022,100.580822
3,2021-07-07,98.292611,98.727057,98.769491,99.545850,102.212189,100.942084,101.844012,101.799360,100.936260
4,2021-07-08,96.489965,97.525438,97.620875,98.384701,101.250419,99.997927,101.152868,101.184541,100.113842
5,2021-07-09,97.932481,99.162984,99.575815,100.048404,102.263651,101.380836,101.960185,101.816299,101.182521
6,2021-07-12,98.508912,99.618838,100.100626,100.416663,102.093165,101.632426,102.333279,102.213960,101.544968
7,2021-07-13,97.974909,98.793685,99.151743,99.647714,102.057221,101.170558,102.330170,102.213960,101.198808
8,2021-07-14,97.081468,98.416136,98.649584,99.638837,101.864828,100.942390,102.681860,102.397266,101.349813
9,2021-07-15,95.901880,98.141087,98.412396,99.576344,101.882347,100.853176,102.223172,101.678099,101.003668


## 4. Quantitative Analytics & Multi-Benchmark Performance Matrix (2021–2026)

In [4]:
def compute_strategy_analytics(series, spy_series, rf=0.02):
    daily_rets = series.pct_change().dropna()
    spy_rets = spy_series.pct_change().dropna()
    aligned = pd.concat([daily_rets, spy_rets], axis=1, join='inner').dropna()
    r_strat, r_spy = aligned.iloc[:, 0], aligned.iloc[:, 1]
    
    n_years = len(r_strat) / 252.0
    total_ret = (series.iloc[-1] / series.iloc[0]) - 1.0
    cagr = (series.iloc[-1] / series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0
    ann_excess = (r_strat.mean() * 252.0) - rf
    ann_vol = r_strat.std() * np.sqrt(252.0)
    sharpe = ann_excess / ann_vol if ann_vol > 0 else 0.0
    downside_vol = r_strat[r_strat < 0].std() * np.sqrt(252.0)
    sortino = ann_excess / downside_vol if downside_vol > 0 else 0.0
    
    drawdown = (series - series.cummax()) / series.cummax()
    max_dd = drawdown.min()
    calmar = cagr / abs(max_dd) if abs(max_dd) > 0 else 0.0
    cov_matrix = np.cov(r_strat, r_spy)
    beta = cov_matrix[0, 1] / cov_matrix[1, 1] if cov_matrix[1, 1] > 0 else 1.0
    alpha = (cagr - rf) - beta * (((spy_series.iloc[-1] / spy_series.iloc[0]) ** (1.0 / max(n_years, 0.1)) - 1.0) - rf)
    
    return {
        'Total Return (%)': total_ret * 100.0,
        'CAGR (%)': cagr * 100.0,
        'Sharpe Ratio': sharpe,
        'Sortino Ratio': sortino,
        'Max Drawdown (%)': max_dd * 100.0,
        'Calmar Ratio': calmar,
        'Market Beta (β)': beta,
        'Jensen Alpha (α %)': alpha * 100.0
    }

eval_list = [
    ('Unified Engine (Concentrated Top 10 Alpha)', df_master_eval['Strategy_Unified_Concentrated_Top10']),
    ('Unified Engine (Diversified Top 100)', df_master_eval['Strategy_Unified_Diversified_Top100']),
    ('Multi-Modal Alpha (Diversified Top 100)', df_master_eval['Strategy_MultiModal_Diversified_Top100']),
    ('Baseline Naive XGBoost (Diversified Top 100)', df_master_eval['Strategy_Baseline_Diversified_Top100']),
    ('Top 10 US Equities Static Buy & Hold', df_master_eval['Benchmark_CandidatePool_Top10_BH']),
    ('Top 100 US Equities Static Buy & Hold', df_master_eval['Benchmark_CandidatePool_Top100_BH']),
    ('S&P 100 Index (XLG - 100% US Mega-Cap)', df_master_eval['Benchmark_XLG_SP100']),
    ('NASDAQ 100 Index (QQQ - Tech/Growth Benchmark)', df_master_eval['Benchmark_QQQ_Nasdaq100']),
    ('S&P 500 Index (SPY - Broad Market Benchmark)', df_master_eval['Benchmark_SPY_SP500'])
]

summary_stats = []
for name, s in eval_list:
    summary_stats.append({'Strategy / Benchmark': name, **compute_strategy_analytics(s, df_master_eval['Benchmark_SPY_SP500'])})

df_analytics_table = pd.DataFrame(summary_stats)
print("=== INSTITUTIONAL STRATEGY & MULTI-INDEX BENCHMARK MATRIX (2021-2026) ===")
df_analytics_table

=== INSTITUTIONAL STRATEGY & MULTI-INDEX BENCHMARK MATRIX (2021-2026) ===


,Strategy / Benchmark,Total Return (%),CAGR (%),Sharpe Ratio,Sortino Ratio,Max Drawdown (%),Calmar Ratio,Market Beta (β),Jensen Alpha (α %)
0,Unified Engine (Concentrated Top 10 Alpha),220.196560,25.437490,1.114783,1.531083,-19.136426,1.329271,0.785749,14.394636
1,Unified Engine (Diversified Top 100),85.565746,12.794676,0.861173,1.206973,-16.540270,0.773547,0.550365,4.460757
2,Multi-Modal Alpha (Diversified Top 100),100.883404,14.550459,0.818948,1.141539,-19.574090,0.743353,0.781898,3.551924
3,Baseline Naive XGBoost (Diversified Top 100),134.639777,18.068449,0.852939,1.209374,-23.034218,0.784418,1.020767,4.320862
4,Top 10 US Equities Static Buy & Hold,27.630259,4.865816,0.239944,0.342296,-30.778632,0.158091,0.917878,-7.697659
5,Top 100 US Equities Static Buy & Hold,64.318738,10.154940,0.541496,0.778462,-25.141199,0.403916,0.930457,-2.553303
6,S&P 100 Index (XLG - 100% US Mega-Cap),100.278569,14.483210,0.709369,0.998325,-28.016995,0.516944,1.068971,0.180867
7,NASDAQ 100 Index (QQQ - Tech/Growth Benchmark),109.560188,15.497683,0.660004,0.943663,-35.118713,0.441294,1.265849,-1.070446
8,S&P 500 Index (SPY - Broad Market Benchmark),91.676152,13.508586,0.711938,0.985087,-24.496389,0.551452,1.000000,0.000000


## 5. Full-Width Interactive Equity Curves & Underwater Drawdowns (2021–2026)

In [5]:
fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08,
                    subplot_titles=('<b>Unified Alpha Engine (Top 10 vs. Top 100) vs. Top 10 B&H, S&P 100 (XLG), NASDAQ 100 (QQQ) & S&P 500 (SPY)</b>',
                                    '<b>Underwater Drawdown Curves (%)</b>'))

colors = {
    'Strategy_Unified_Concentrated_Top10': '#00CC96',
    'Strategy_Unified_Diversified_Top100': '#00B4D8',
    'Strategy_MultiModal_Diversified_Top100': '#3366CC',
    'Strategy_Baseline_Diversified_Top100': '#AB63FA',
    'Benchmark_CandidatePool_Top10_BH': '#FECB52',
    'Benchmark_CandidatePool_Top100_BH': '#FFA15A',
    'Benchmark_XLG_SP100': '#B6E880',
    'Benchmark_QQQ_Nasdaq100': '#FF6692',
    'Benchmark_SPY_SP500': '#636EFA'
}

labels = {
    'Strategy_Unified_Concentrated_Top10': 'Unified Alpha Engine (Concentrated Top 10)',
    'Strategy_Unified_Diversified_Top100': 'Unified Engine (Diversified Top 100)',
    'Strategy_MultiModal_Diversified_Top100': 'Multi-Modal Alpha (Top 100)',
    'Strategy_Baseline_Diversified_Top100': 'Baseline Naive XGBoost (Top 100)',
    'Benchmark_CandidatePool_Top10_BH': 'Top 10 US Stocks Static B&H',
    'Benchmark_CandidatePool_Top100_BH': 'Top 100 US Stocks Static B&H',
    'Benchmark_XLG_SP100': 'S&P 100 (XLG - 100% US Mega-Caps)',
    'Benchmark_QQQ_Nasdaq100': 'NASDAQ 100 (QQQ Benchmark)',
    'Benchmark_SPY_SP500': 'S&P 500 (SPY Benchmark)'
}

for col, name in labels.items():
    s = df_master_eval[col]
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=s, name=name,
        line=dict(color=colors[col], width=3.5 if 'Concentrated' in name else (2.5 if 'Benchmark' in col else 1.8))
    ), row=1, col=1)
    
    dd = ((s - s.cummax()) / s.cummax()) * 100.0
    fig.add_trace(go.Scatter(
        x=df_master_eval['date'], y=dd, name=f"{name} DD", showlegend=False,
        line=dict(color=colors[col], width=1.5)
    ), row=2, col=1)

fig.update_layout(
    template='plotly_dark', width=1100, height=750,
    title='<b>Concentrated Alpha vs. Broad Diversification vs. Market Indexes (2021-2026)</b>',
    margin=dict(l=60, r=230, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Benchmark</b>'))
)
fig.show()

## 6. Annual Returns & Profit Breakdown by Year (Bar Chart 2021–2026)

In [6]:
# Compute annual returns per year
df_master_eval['year'] = df_master_eval['date'].dt.year
strat_cols = [c for c in df_master_eval.columns if c not in ['date', 'year']]

annual_records = []
for y, grp in df_master_eval.groupby('year'):
    year_label = f"{y} (H2)" if y == 2021 else (f"{y} (YTD)" if y == 2026 else str(y))
    start_vals = grp.iloc[0][strat_cols]
    end_vals = grp.iloc[-1][strat_cols]
    rets = ((end_vals / start_vals) - 1.0) * 100.0
    record = {'Year': year_label}
    for col in strat_cols:
        record[labels[col]] = rets[col]
    annual_records.append(record)

df_annual_table = pd.DataFrame(annual_records)
print("=== ANNUAL RETURNS BREAKDOWN BY YEAR (% PROFIT) ===")
print(df_annual_table.to_string(index=False))

# Build Grouped Bar Chart
fig_bar = go.Figure()

for col in strat_cols:
    lbl = labels[col]
    fig_bar.add_trace(go.Bar(
        x=df_annual_table['Year'],
        y=df_annual_table[lbl],
        name=lbl,
        marker_color=colors[col],
        text=df_annual_table[lbl].apply(lambda v: f"{v:+.1f}%"),
        textposition='outside'
    ))

fig_bar.update_layout(
    template='plotly_dark',
    barmode='group',
    width=1150,
    height=600,
    title='<b>Annual Profit / Return (%) by Year (2021–2026): Strategies vs. Benchmarks</b>',
    xaxis=dict(title='<b>Trading Year</b>', tickfont=dict(size=13, family='Arial')),
    yaxis=dict(title='<b>Annual Return (%)</b>', tickfont=dict(size=12), zeroline=True, zerolinewidth=1.5, zerolinecolor='gray'),
    margin=dict(l=60, r=230, t=80, b=60),
    legend=dict(orientation='v', yanchor='top', y=1.0, xanchor='left', x=1.02, title=dict(text='<b>Strategy / Benchmark</b>'))
)
fig_bar.show()

=== ANNUAL RETURNS BREAKDOWN BY YEAR (% PROFIT) ===
      Year  Unified Alpha Engine (Concentrated Top 10)  Unified Engine (Diversified Top 100)  Multi-Modal Alpha (Top 100)  Baseline Naive XGBoost (Top 100)  Top 10 US Stocks Static B&H  Top 100 US Stocks Static B&H  S&P 100 (XLG - 100% US Mega-Caps)  NASDAQ 100 (QQQ Benchmark)  S&P 500 (SPY Benchmark)
 2021 (H2)                                   -0.231511                              0.118794                     8.534603                         11.213001                     7.915248                     10.776941                          13.123901                   12.474221                11.087871
      2022                                    1.498119                              4.316232                    -2.064822                         -6.118769                   -15.092406                    -14.289500                         -25.214867                  -33.219864               -18.646389
      2023                             

## 7. Export Results to Excel

In [7]:
output_inst_path = os.path.join(LOCAL_DATA_DIR, "institutional_strategy_comparison_poc.xlsx")
with pd.ExcelWriter(output_inst_path) as writer:
    df_master_eval.drop(columns=['year']).to_excel(writer, sheet_name='daily_equity_curves', index=False)
    df_analytics_table.to_excel(writer, sheet_name='summary_metrics', index=False)
    df_annual_table.to_excel(writer, sheet_name='annual_returns_by_year', index=False)

print(f"💾 Successfully exported Institutional Strategy simulations & Annual Returns to: {output_inst_path}")

💾 Successfully exported Institutional Strategy simulations & Annual Returns to: c:\Users\honza\Desktop\projects\stock-analysis\research\notebooks\algo-alpha-execution\data\fetched\institutional_strategy_comparison_poc.xlsx
